# 🛒 Analyse RFM — Segmentation Clients E-commerce

**Dataset** : UCI Online Retail (541 909 transactions, UK 2010-2011)  
**Objectif** : calculer les scores RFM pour chaque client et préparer la segmentation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

# ─── Données simulées (fidèles à l'UCI Online Retail) ────────────────────
# En production : pd.read_excel('data/Online Retail.xlsx')
N = 10000
customer_ids = np.random.choice(range(10000, 18500), N)
dates = pd.date_range('2010-12-01', '2011-12-09', periods=N)
dates = pd.Series(dates).sample(frac=1, random_state=42).values

df = pd.DataFrame({
    'InvoiceNo': [f'5{i:05d}' for i in range(N)],
    'CustomerID': customer_ids,
    'InvoiceDate': dates,
    'Quantity': np.random.choice(range(1, 30), N, p=np.ones(29)/29),
    'UnitPrice': np.round(np.random.lognormal(2.5, 1.2, N), 2).clip(0.1, 200),
    'Country': np.random.choice(['United Kingdom','Germany','France','Spain'],
                                 N, p=[0.85, 0.07, 0.05, 0.03])
})
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df = df[df['TotalPrice'] > 0]

print(f'Dataset : {len(df):,} transactions · {df.CustomerID.nunique():,} clients uniques')
df.head()

## 1. Nettoyage des données

In [ ]:
print(f'Valeurs manquantes :\n{df.isnull().sum()}')
print(f'Doublons : {df.duplicated().sum()}')
df = df.dropna(subset=['CustomerID'])
df['CustomerID'] = df['CustomerID'].astype(int)
print(f'\nDataset nettoyé : {len(df):,} transactions · {df.CustomerID.nunique():,} clients')

## 2. Calcul des scores RFM

In [ ]:
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum')
).reset_index()

# Scores quintiles (1 = faible, 5 = fort)
rfm['R_score'] = pd.qcut(rfm['Recency'], q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['Monetary'], q=5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

print(rfm.describe().round(2))

## 3. Distribution des scores RFM

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
couleurs = ['#E74C3C', '#3498DB', '#2ECC71']
for ax, col, titre, c in zip(axes, ['Recency','Frequency','Monetary'],
                               ['Récence (jours)','Fréquence (commandes)','Valeur (£)'],
                               couleurs):
    ax.hist(rfm[col], bins=40, color=c, alpha=0.8, edgecolor='white')
    ax.axvline(rfm[col].median(), color='black', linestyle='--', label=f'Médiane: {rfm[col].median():.0f}')
    ax.set_title(titre, fontweight='bold', fontsize=12)
    ax.legend()

plt.suptitle('Distribution des métriques RFM', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('rfm_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

rfm.to_csv('data/rfm_scores.csv', index=False)
print('✅ Scores RFM sauvegardés → data/rfm_scores.csv')